# Task D — identification sweep and prior overlay

Pre-registered in `DECISIONS.md` §14. Per prior: train with the frozen recipe (`python -m taskc.run_rung --tag <tag> --prior <tag> --epochs 450 --lr_decay --ema --device mps`), gate against the prior's own simulator (`taskc_02_gate.ipynb` with `RUN_TAG=<tag>`), then `python -m taskc.sweep --tag <tag>` (10⁶ solve draw, C0→C3, numbers on draw C). This notebook reads the `sweep.json` files and draws the identification figure: x = constraint level, y = exotic price, one band per prior; retrains define the training-variance band.

Pre-registered ordering at C3: wrong μ smallest spread, wrong ρ largest, wrong κ intermediate. Cell 0 is the Colab cell; locally skip it.

In [ ]:
%cd /content
%rm -rf ddpm_option_pricing
!git clone https://github.com/nilay47/ddpm_option_pricing.git
%cd ddpm_option_pricing
!git fetch --all
!git checkout v2_code
!git status

In [ ]:
import os, sys, json
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
import numpy as np, matplotlib.pyplot as plt
import taskc
from taskc.config import CFG
from taskc.dual import TASKB_EXOTICS_Q, TASKB_EXOTICS_P, TASKB
from config import EXOTICS
PRIORS = [("lrema", "base"), ("retrain1", "retrain 1"), ("retrain2", "retrain 2"), ("mu25", "mu=0.25"), ("rho02", "rho=-0.2"), ("kappa6", "kappa=6")]
LEVELS = ["C0", "C1", "C2", "C3"]
sw = {}
for tag, lab in PRIORS:
    p = os.path.join(CFG.run_dir(tag), "sweep", "sweep.json")
    if os.path.exists(p): sw[tag] = json.load(open(p))
    else: print("missing:", p)
print("loaded:", list(sw))

## Per-prior sweep tables

In [ ]:
for tag, lab in PRIORS:
    if tag not in sw: continue
    s = sw[tag]; print(f"=== {lab} ({tag})  heston={s['heston']}  levels S6 {s['level_S6']:+.4f} C {s['level_C']:+.4f}  CV(RV) {s['sv_C']['cv_rv']:.3f} lev5 {s['sv_C']['lev5']:+.3f}")
    print(f"{'lvl':4s} {'scr':>4s} {'margin':>7s} {'|b_raw|':>8s} {'ESS_S6%':>8s} {'ESS_C%':>7s} {'KL':>7s} {'hoVan_in':>9s} {'hoVan_C':>8s} {'hoMart_C':>9s} " + " ".join(f"{k[:10]:>16s}" for k in EXOTICS))
    for lvl in LEVELS:
        L = s["levels"][lvl]
        print(f"{lvl:4s} {'ok' if L['screen_ok'] else 'FAIL':>4s} {L['screen_margin']:7.3f} {L['beta_raw_norm']:8.2f} {L['ess_solve']*100:8.2f} {L['ess_C']*100:7.2f} {L['kl']:7.4f} {L['ho_van_in']:9.4f} {L['ho_van_C']:8.4f} {L['ho_mart_C']:9.2e} "
              + " ".join(f"{L['exotics_C'][k][0]:8.4f}+-{L['exotics_C'][k][1]:.4f}" for k in EXOTICS))
    print()

## The identification figure: exotic price vs constraint level, one line per prior

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
x = np.arange(len(LEVELS) + 1)
for axi, k in zip(ax, EXOTICS):
    for tag, lab in PRIORS:
        if tag not in sw: continue
        s = sw[tag]; v = [s["unweighted_C"]["exotics"][k][0]] + [s["levels"][l]["exotics_C"][k][0] for l in LEVELS]
        e = [s["unweighted_C"]["exotics"][k][1]] + [s["levels"][l]["exotics_C"][k][1] for l in LEVELS]
        ls = "-" if tag == "lrema" else ("--" if tag.startswith("retrain") else "-."); lw = 2 if tag == "lrema" else 1.2
        axi.errorbar(x, v, yerr=[2*t for t in e], marker="o", ms=3, ls=ls, lw=lw, capsize=2, label=lab)
    axi.axhline(TASKB_EXOTICS_Q[k], c="k", ls=":", lw=1, label="Heston-Q"); axi.axhline(TASKB_EXOTICS_P[k], c="grey", ls=":", lw=1, label="Heston-P")
    axi.set_xticks(x); axi.set_xticklabels(["P_theta"] + LEVELS); axi.set_title(k); axi.grid(alpha=.3)
ax[0].legend(fontsize=7); plt.tight_layout(); plt.show()

## Spread at C3 per prior vs the retrain band (pre-registered ordering: μ < κ < ρ)

In [ ]:
if "lrema" in sw:
    b = sw["lrema"]
    print(f"{'prior':10s} " + " ".join(f"{k[:12]:>24s}" for k in EXOTICS) + f" {'ESS_C3%':>8s} {'|b_raw|_C0':>10s}")
    for tag, lab in PRIORS:
        if tag not in sw: continue
        s = sw[tag]; row = f"{lab:10s} "
        for k in EXOTICS:
            d = s["levels"]["C3"]["exotics_C"][k][0] - b["levels"]["C3"]["exotics_C"][k][0]
            se = np.hypot(s["levels"]["C3"]["exotics_C"][k][1], b["levels"]["C3"]["exotics_C"][k][1])
            span = TASKB_EXOTICS_P[k] - TASKB_EXOTICS_Q[k]
            row += f" {d:+8.4f} ({d/se:+5.1f}SE, {d/span*100:+5.0f}% span)"
        print(row + f" {s['levels']['C3']['ess_C']*100:8.2f} {s['levels']['C0']['beta_raw_norm']:10.2f}")
    band = {k: max(abs(sw[t]["levels"]["C3"]["exotics_C"][k][0] - b["levels"]["C3"]["exotics_C"][k][0]) for t in ("retrain1", "retrain2") if t in sw) for k in EXOTICS} if any(t in sw for t in ("retrain1", "retrain2")) else None
    print("\ntraining-variance band (max |retrain - base| at C3):", band)

Results and the ordering verdict go to `DECISIONS.md` §15.